<a href="https://colab.research.google.com/github/extreme500/Laborat-rio-de-Visualiza-o-CGVis/blob/main/Gr%C3%A1fico%20de%20Dispers%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import plotly.express as px
import numpy as np
from scipy.stats import gaussian_kde

df = pd.read_csv("Gaming_Academic_Performance.csv")

df_filtrado = df[
    (df["grades"] <= 100)
].copy()

df_melted = df_filtrado.melt(
    id_vars=["grades"],
    value_vars=["sleep_hours", "gaming_hours", "study_hours"],
    var_name="variable",
    value_name="hours"
)

df_melted["variable"] = df_melted["variable"].replace({
    "sleep_hours": "Horas de sono",
    "gaming_hours": "Horas de jogo",
    "study_hours": "Horas de estudo"
})

# Calculate density for each facet
density_list = []
for var in df_melted['variable'].unique():
    subset = df_melted[df_melted['variable'] == var]
    x = subset['hours']
    y = subset['grades']
    if len(x) > 1 and len(y) > 1: # Ensure there's enough data for KDE
        xy = np.vstack([x, y])
        z = gaussian_kde(xy)(xy)
    else:
        z = np.zeros(len(x)) # Assign zero density if not enough data

    density_list.append(pd.DataFrame({
        'grades': y,
        'variable': var,
        'hours': x,
        'density': z
    }))

df_melted_with_density = pd.concat(density_list)

fig = px.scatter(
    df_melted_with_density,
    x="hours",
    y="grades",
    facet_col="variable",
    color="density", # Use density for color
    color_continuous_scale="Viridis", # A good scale for density
    opacity=0.7,
    trendline="ols",
    facet_col_spacing=0.08,
    labels={
        "grades": "Notas",
        "hours": "Horas",
        "density": "Densidade" # Update label for density
        #
    },
    title="Relação entre Hábitos e Notas (Colorido por Densidade)", # Update title
    hover_data=['variable'] # Add 'variable' to hover info
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_yaxes(title_text="Notas", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=2)
fig.update_yaxes(title_text="", row=1, col=3)

fig.update_layout(
    template="plotly_white",
    margin=dict(l=60, r=40, t=60, b=40)
)

fig.add_shape(
    type="line",
    x0=0.33, x1=0.33,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(color="black", width=1)
)

fig.add_shape(
    type="line",
    x0=0.66, x1=0.66,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(color="black", width=1)
)

fig.show()